In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import optuna

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from sklearn.metrics import root_mean_squared_error

from rose.optimize import (
    plot_categorical_hyperparameter,
    plot_hyperparameter_trials_3d,
    plot_numeric_hyperparameter,
)
from rose.viz import plot_response_surface

In [2]:
N_TRIALS = 100

ACTIVATIONS: dict[str, type[nn.Module]] = {
    "relu": nn.ReLU,
    "gelu": nn.GELU,
    "silu": nn.SiLU, # swish
    "sigmoid": nn.Sigmoid,
}

In [3]:
def training_data(N: int = 10, seed: int = 37):
    np.random.seed(seed)
    X = np.random.randint(1, 7, size=(N, 5)).astype(float)
    X_petals = np.where(X == 3, 2, 0) + np.where(X == 5, 4, 0)
    y = X_petals.sum(axis=1).astype(float)

    return X, y

X_train, y_train = training_data(1_000, seed=37)
X_valid, y_valid = training_data(1_000, seed=42)

In [4]:
class FCNet(nn.Module):
    def __init__(
        self,
        hidden_1: int,
        hidden_2: int,
        activation: type[nn.Module],
    ):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(5, hidden_1),
            nn.BatchNorm1d(hidden_1),
            activation(),
            nn.Linear(hidden_1, hidden_2),
            nn.BatchNorm1d(hidden_2),
            activation(),
            nn.Linear(hidden_2, 1),
        )

    def forward(self, X):
        return self.net(X)


def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.normal_(m.weight, mean=0.0, std=0.1)
        nn.init.constant_(m.bias, 0.0)

In [5]:
def calculate_metrics(y_true: np.ndarray, y_hat: np.ndarray):
    y_prediction = np.rint(y_hat)
    return {
        "rmse": root_mean_squared_error(y_true, y_hat),
        "mae": np.mean(np.abs(y_hat - y_true)),
        "bias": np.mean(y_hat - y_true),
        "accuracy": np.mean(y_prediction == y_true),
        "accuracy_pm1": np.mean(np.abs(y_prediction - y_true) <= 1),
    }

In [6]:
class FCNNRegressor:
    def __init__(
        self,
        *,
        hidden_1: int = 100,
        hidden_2: int = 10,
        learning_rate: float = 0.01,
        activation: type[nn.Module] = nn.ReLU,
        batch_size: int = 1024,
        epochs: int = 2_000,
        seed: int = 37,
    ):
        self.hidden_1 = hidden_1
        self.hidden_2 = hidden_2
        self.learning_rate = learning_rate
        self.activation = activation
        self.batch_size = batch_size
        self.epochs = epochs
        self.seed = seed
        self.device = "cpu" # torch.device("cuda" if torch.cuda.is_available() else "cpu")

        self.model = None
        self.history = []

    @staticmethod
    def _init_weights(module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.1)
            nn.init.constant_(module.bias, 0.0)

    def fit(self, X: np.ndarray, y: np.ndarray):
        self.history = []

        torch.manual_seed(self.seed)

        X_tensor = torch.tensor(X, dtype=torch.float32).to(self.device)
        y_tensor = torch.tensor(y, dtype=torch.float32).reshape(-1, 1).to(self.device)

        self.model = FCNet(
            hidden_1=self.hidden_1,
            hidden_2=self.hidden_2,
            activation=self.activation,
        ).to(self.device)
        self.model.apply(self._init_weights)

        optimizer = optim.Adam(self.model.parameters(), lr=self.learning_rate)
        loss_function = nn.MSELoss()

        train_dataset = TensorDataset(X_tensor, y_tensor)
        train_loader = DataLoader(
            train_dataset,
            batch_size=self.batch_size,
            shuffle=True
        )
        
        for epoch in range(self.epochs):
            # train
            self.model.train()
            epoch_losses = []
            for X_batch, y_batch in train_loader:
                optimizer.zero_grad()
                y_hat = self.model(X_batch)
                loss = loss_function(y_hat, y_batch)
                loss.backward()
                optimizer.step()
                epoch_losses.append(float(loss.detach().cpu()))

            # evaluate
            self.model.eval()
            with torch.no_grad():
                y_hat = self.model(X_tensor).detach().cpu().numpy().ravel()

            # log metrics
            self.history.append({
                "epoch": epoch,
                "loss": np.mean(epoch_losses),
                **calculate_metrics(y, y_hat),
            })

        return self

    def predict(self, X: np.ndarray | torch.Tensor) -> np.ndarray:
        if self.model is None:
            raise RuntimeError("Model must be fit before calling predict().")
    
        self.model.eval()
    
        if isinstance(X, np.ndarray):
            X_tensor = torch.tensor(X, dtype=torch.float32).to(self.device)
        elif isinstance(X, torch.Tensor):
            X_tensor = X.to(device=self.device, dtype=torch.float32)
        else:
            raise TypeError("X must be a numpy array or torch tensor.")
    
        with torch.no_grad():
            y_hat = self.model(X_tensor)
    
        return y_hat.detach().cpu().numpy().ravel()

In [7]:
def objective(trial):
    hidden_1 = trial.suggest_int("hidden_1", 8, 1024)
    hidden_2 = trial.suggest_int("hidden_2", 1, min(256, hidden_1))
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-1, log=True)
    batch_size = trial.suggest_categorical("batch_size", [64, 128, 256, 512, 1024])
    activation_name = trial.suggest_categorical("activation", ["relu", "gelu", "silu", "sigmoid"])

    model = FCNNRegressor(
        hidden_1=hidden_1,
        hidden_2=hidden_2,
        learning_rate=learning_rate,
        batch_size=batch_size,
        activation=ACTIVATIONS[activation_name],
        epochs=2_000,
    )
    model.fit(X_train, y_train)

    y_hat = model.predict(X_valid)
    metrics = calculate_metrics(y_valid, y_hat)
    for metric, value in metrics.items():
        trial.set_user_attr(metric, value)
    
    return metrics["rmse"]

In [8]:
study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=37),
)

[I 2026-05-25 19:55:56,446] A new study created in memory with name: no-name-2a5cc97e-c009-4393-ab8f-cc89e0cbc3bf


In [ ]:
study.optimize(objective, n_trials=N_TRIALS)

print("Best parameters:")
print(study.best_params)

print("\nBest RMSE:")
print(study.best_value)

print("\nBest accuracy:")
print(study.best_trial.user_attrs["accuracy"])

[I 2026-05-25 19:56:14,898] Trial 0 finished with value: 2.6382566172662387 and parameters: {'hidden_1': 968, 'hidden_2': 119, 'learning_rate': 0.0003787782967895686, 'batch_size': 1024, 'activation': 'silu'}. Best is trial 0 with value: 2.6382566172662387.


In [ ]:
print("Best parameters:")
print(study.best_params)

print("\nBest RMSE:")
print(study.best_value)

print("\nBest accuracy:")
print(study.best_trial.user_attrs["accuracy"])

In [ ]:
def train_champion(
    study,
    *,
    N: int = 1000,
) -> FCNNRegressor:
    params = study.best_params

    X_train, y_train = training_data(N, seed=37)

    model = FCNNRegressor(
        hidden_1=params["hidden_1"],
        hidden_2=params["hidden_2"],
        learning_rate=params["learning_rate"],
        batch_size=params["batch_size"],
        activation=ACTIVATIONS[params["activation"]],
    )

    model.fit(X_train, y_train)

    return model


model = train_champion(study)

In [ ]:
def plot_history():
    history_df = pd.DataFrame(model.history)

    plt.figure(figsize=(12, 5))
    plt.plot(history_df["epoch"], history_df["loss"], linewidth=1)
    plt.xlabel("Epoch")
    plt.ylabel("Training Loss")
    plt.title("FC NN Training History")
    plt.grid(True, alpha=0.3)
    plt.show()
    
plot_history()

In [ ]:
def evaluate_model(model: FCNNRegressor,*,test_N: int = 10_000):
    X_test, y_test = training_data(test_N, seed=42)
    y_hat = model.predict(X_test)
    return calculate_metrics(y_test, y_hat)
    
evaluate_model(model)

In [ ]:
def plot_model_scatter(
    model,
    *,
    test_N: int = 200,
):
    X_test, y_test = training_data(test_N, seed=42)
    y_hat = model.predict(X_test)
    metrics = calculate_metrics(y_test, y_hat)

    # scatter plot
    alpha = min(1.0, 20 / test_N)
    label = (
        f"RMSE = {metrics['rmse']:.3f}"
#        f"Accuracy = {100 * metrics['accuracy']:.1f}%\n"
#        f"Accuracy ±1 = {100 * metrics['accuracy_pm1']:.1f}%"
    )
    plt.scatter(y_test, y_hat, alpha=alpha, edgecolors="none", label=label)

    # reference diagonal
    plt.axline(
        (0, 0),
        slope=1,
        linestyle="--",
        color="gray",
        alpha=0.5,
        lw=1,
    )

    plt.xlim(0, 20)
    plt.ylim(0, 20)

    # chart junk
    plt.xlabel("Actual")
    plt.ylabel("Predicted")
    plt.legend(loc="lower right")
    plt.title("FC NN: Actual vs. Predicted")

    plt.gca().set_aspect("equal", adjustable="box")
    plt.grid(True, alpha=0.3)

plot_model_scatter(model)

In [ ]:
trials_df = study.trials_dataframe()

for hyperparameter in [
    "params_hidden_1",
    "params_hidden_2",
    "params_learning_rate",
    "params_batch_size",
]:
    plot_numeric_hyperparameter(
        trials_df,
        hyperparameter,
        log_x=(hyperparameter == "params_learning_rate"),
        bandwidth_fraction=0.1,
        remove_bottom=0.2,
    )

In [ ]:
study.trials_dataframe().columns

In [ ]:
trials_df = study.trials_dataframe()

plot_categorical_hyperparameter(
    trials_df,
    "params_activation",
    target="user_attrs_accuracy",
    greater_is_better=True,
)

plot_categorical_hyperparameter(
    trials_df,
    "params_activation",
    target="user_attrs_rmse",
)

In [ ]:
plot_hyperparameter_trials_3d(
    study.trials_dataframe(),
    "params_hidden_1",
    "params_hidden_2",
    title="FC NN Hyperparameter Trials",
)

In [ ]:
plot_response_surface(
    model,
    feature_count=5,
    fixed_values=(1, 1, 1),
    x_index=0,
    y_index=1,
    title="Model Response Surface",
    colorscale="Viridis",
)